In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# ==============================================================================
# Data Generation code is taken from Google Gemini
#
# 1. PLAYER PROFILES (60 Players: 4 Teams * 15 Players)
# ==============================================================================
teams = {
    "India": [
        ("Rohit Sharma", "Batsman"), ("Shubman Gill", "Batsman"), ("Virat Kohli", "Batsman"),
        ("Shreyas Iyer", "Batsman"), ("KL Rahul", "Batsman"), ("Hardik Pandya", "All-Rounder"),
        ("Ravindra Jadeja", "All-Rounder"), ("Kuldeep Yadav", "Bowler"), ("Mohammed Shami", "Bowler"),
        ("Jasprit Bumrah", "Bowler"), ("Mohammed Siraj", "Bowler"),
        ("Ishan Kishan", "Batsman"), ("Suryakumar Yadav", "All-Rounder"), ("R. Ashwin", "All-Rounder"), ("Shardul Thakur", "Bowler")
    ],
    "Australia": [
        ("David Warner", "Batsman"), ("Travis Head", "Batsman"), ("Mitchell Marsh", "All-Rounder"),
        ("Steve Smith", "Batsman"), ("Marnus Labuschagne", "Batsman"), ("Glenn Maxwell", "All-Rounder"),
        ("Josh Inglis", "Batsman"), ("Pat Cummins", "Bowler"), ("Mitchell Starc", "Bowler"),
        ("Adam Zampa", "Bowler"), ("Josh Hazlewood", "Bowler"),
        ("Cameron Green", "All-Rounder"), ("Alex Carey", "Batsman"), ("Marcus Stoinis", "All-Rounder"), ("Sean Abbott", "Bowler")
    ],
    "England": [
        ("Jonny Bairstow", "Batsman"), ("Dawid Malan", "Batsman"), ("Joe Root", "Batsman"),
        ("Ben Stokes", "All-Rounder"), ("Jos Buttler", "Batsman"), ("Liam Livingstone", "All-Rounder"),
        ("Moeen Ali", "All-Rounder"), ("Chris Woakes", "Bowler"), ("David Willey", "Bowler"),
        ("Adil Rashid", "Bowler"), ("Mark Wood", "Bowler"),
        ("Harry Brook", "Batsman"), ("Sam Curran", "All-Rounder"), ("Reece Topley", "Bowler"), ("Gus Atkinson", "Bowler")
    ],
    "Pakistan": [
        ("Abdullah Shafique", "Batsman"), ("Fakhar Zaman", "Batsman"), ("Babar Azam", "Batsman"),
        ("Mohammad Rizwan", "Batsman"), ("Saud Shakeel", "Batsman"), ("Iftikhar Ahmed", "All-Rounder"),
        ("Shadab Khan", "All-Rounder"), ("Usama Mir", "Bowler"), ("Shaheen Afridi", "Bowler"),
        ("Haris Rauf", "Bowler"), ("Hasan Ali", "Bowler"),
        ("Imam-ul-Haq", "Batsman"), ("Agha Salman", "Batsman"), ("Mohammad Nawaz", "All-Rounder"), ("Mohammad Wasim", "Bowler")
    ]
}

player_profiles = []
player_id_map = {}
p_id_counter = 100

for team, players in teams.items():
    for name, role in players:
        p_id = f"{team[:3].upper()}_{p_id_counter}"
        age = random.randint(22, 36)
        player_profiles.append({
            "PlayerID": p_id, "PlayerName": name, "Team": team,
            "PlayingRole": role, "AgeProfile": f"{age} yrs"
        })
        player_id_map[name] = p_id
        p_id_counter += 1

df_players = pd.DataFrame(player_profiles)

# ==============================================================================
# 2. MATCH SUMMARY (50 Matches)
# ==============================================================================
venues = ["Wankhede Stadium", "MCG", "Lords", "Gaddafi Stadium", "Eden Gardens", "SCG"]
teams_list = list(teams.keys())
matches = []
match_counter = 1001

start_date = datetime(2023, 1, 1)

for _ in range(50):
    t1, t2 = random.sample(teams_list, 2)
    match_id = f"M_{match_counter}"
    date = (start_date + timedelta(days=random.randint(1, 150))).strftime("%Y-%m-%d")
    venue = random.choice(venues)
    toss_won = random.choice([t1, t2])
    toss_dec = random.choice(["Bat", "Bowl"])
    winner = random.choice([t1, t2])
    margin = f"{random.randint(10, 80)} runs" if random.choice([True, False]) else f"{random.randint(3, 8)} wickets"

    cap1 = teams[t1][2][0] # 3rd player usually cap

    matches.append({
        "MatchID": match_id, "MatchDate": date, "Venue": venue,
        "CaptainName": cap1, "TossWonTeam": toss_won, "TossDecision": toss_dec,
        "MatchWinner": winner, "WinMargin": margin,
        "Team1": t1, "Team2": t2 # Temporary for loops below
    })
    match_counter += 1

df_matches_full = pd.DataFrame(matches)
df_matches = df_matches_full.drop(columns=['Team1', 'Team2'])

# ==============================================================================
# 3 & 4. BATTING & BOWLING PERFORMANCES (For each match, simulate innings)
# ==============================================================================
batting_data = []
bowling_data = []
innings_id_counter = 5001

for idx, match in df_matches_full.iterrows():
    m_id = match['MatchID']
    t1, t2 = match['Team1'], match['Team2']

    for inn_num, bat_team, bowl_team in [(1, t1, t2), (2, t2, t1)]:
        inn_id = f"INN_{innings_id_counter}"
        innings_id_counter += 1

        # Select Playing 11
        batters = teams[bat_team][:11]
        bowlers = [p for p in teams[bowl_team][:11] if p[1] in ["Bowler", "All-Rounder"]]

        # Batting Stats
        wickets_fallen = 0
        for bat_idx, (b_name, role) in enumerate(batters):
            if wickets_fallen == 10: break

            balls = random.randint(5, 80)
            runs = int(balls * random.uniform(0.5, 1.8))
            fours = runs // 8
            sixes = runs // 16
            sr = round((runs / balls) * 100, 2) if balls > 0 else 0.0

            batting_data.append({
                "InningsID": inn_id, "MatchID": m_id, "PlayerName": b_name,
                "PlayerID": player_id_map[b_name], "Team": bat_team,
                "RunsScored": runs, "BallsFaced": balls,
                "FoursCount": fours, "SixesCount": sixes, "StrikeRate": sr
            })
            if random.random() > 0.3: wickets_fallen += 1

        # Bowling Stats
        overs_to_bowl = 50
        for b_name, role in bowlers:
            if overs_to_bowl <= 0: break

            # --- ERROR FIX: Added max() logic to prevent random.randint(x, y) where x > y ---
            max_limit = min(10, overs_to_bowl)
            if max_limit < 1:
                overs = 0
            else:
                overs = random.randint(1, max_limit) # Changed lower bound to 1 to be safe

            overs_to_bowl -= overs

            if overs > 0:
                runs_conc = int(overs * 6 * random.uniform(0.7, 1.5))
                wkts = random.randint(0, 3)
                econ = round(runs_conc / overs, 2)

                bowling_data.append({
                    "InningsID": inn_id, "MatchID": m_id, "PlayerName": b_name,
                    "PlayerID": player_id_map[b_name], "Team": bowl_team,
                    "OversBowled": overs, "RunsConceded": runs_conc,
                    "WicketsTaken": wkts, "EconomyRate": econ
                })

df_batting = pd.DataFrame(batting_data)
df_bowling = pd.DataFrame(bowling_data)

# ==============================================================================
# SAVE TO CSV
# ==============================================================================
df_players.to_csv("PlayerProfiles.csv", index=False)
df_matches.to_csv("MatchSummary.csv", index=False)
df_batting.to_csv("BattingPerformance.csv", index=False)
df_bowling.to_csv("BowlingPerformance.csv", index=False)

print("Data Generated Successfully! Error Resolved.")
print(f"MatchSummary: {len(df_matches)} rows")
print(f"BattingPerformance: {len(df_batting)} rows")
print(f"BowlingPerformance: {len(df_bowling)} rows")
print(f"PlayerProfiles: {len(df_players)} rows")

Data Generated Successfully! Error Resolved.
MatchSummary: 50 rows
BattingPerformance: 1099 rows
BowlingPerformance: 622 rows
PlayerProfiles: 60 rows


In [2]:
print("--- Batting Table Null Values ---")
print(df_batting.isnull().sum())

print("\n--- Bowling Table Null Values ---")
print(df_bowling.isnull().sum())

print("\n--- Match Summary Null Values ---")
print(df_matches.isnull().sum())

print("\n--- Player Profile Null Values ---")
print(df_players.isnull().sum())

--- Batting Table Null Values ---
InningsID     0
MatchID       0
PlayerName    0
PlayerID      0
Team          0
RunsScored    0
BallsFaced    0
FoursCount    0
SixesCount    0
StrikeRate    0
dtype: int64

--- Bowling Table Null Values ---
InningsID       0
MatchID         0
PlayerName      0
PlayerID        0
Team            0
OversBowled     0
RunsConceded    0
WicketsTaken    0
EconomyRate     0
dtype: int64

--- Match Summary Null Values ---
MatchID         0
MatchDate       0
Venue           0
CaptainName     0
TossWonTeam     0
TossDecision    0
MatchWinner     0
WinMargin       0
dtype: int64

--- Player Profile Null Values ---
PlayerID       0
PlayerName     0
Team           0
PlayingRole    0
AgeProfile     0
dtype: int64
